# U-Net/ResNet18 Training
Main E2 run plus optional E1 resize baseline. Final protocol uses mild photometric augmentation and positive-only Dice loss.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT = Path("/content/TTTN")
assert PROJECT.exists(), "Upload/unzip the project to /content/TTTN first"
assert (PROJECT / "data" / "3cad_ani").exists(), "Dataset missing at /content/TTTN/data/3cad_ani"
%cd /content/TTTN


In [ ]:
!pip install -q -r requirements/ml-kaggle.txt
!python scripts/verification/check_protocol.py


## Main native-resolution patch run
Effective batch = 2 × accumulation 2 = 4. Early stopping uses full-resolution Val Positive Dice@0.5.


In [ ]:
!python scripts/training/train_unet.py \
  --epochs 50 \
  --batch-size 2 \
  --grad-accum 2 \
  --augmentation photometric \
  --run-name main_seed42


## Validation — select threshold only on Val


In [ ]:
!python scripts/evaluation/evaluate_model.py \
  --model unet \
  --checkpoint results/unet_r18/main_seed42/checkpoints/best.pt \
  --split val \
  --warmup-batches 5


## Final Test — do not tune after this


In [ ]:
!python scripts/evaluation/evaluate_model.py \
  --model unet \
  --checkpoint results/unet_r18/main_seed42/checkpoints/best.pt \
  --split test \
  --warmup-batches 5


## Inspect training and Test evidence


In [ ]:
from IPython.display import display, Image
import pandas as pd
base="results/unet_r18/main_seed42"
for f in [
    "curves/learning_curve_loss.png",
    "curves/learning_curve_dice.png",
    "curves/loss_components.png",
    "curves/learning_rate.png",
    "curves/epoch_time.png",
    "curves/vram.png",
    "test/figures/image_roc_curve.png",
    "test/figures/image_pr_curve.png",
    "test/figures/confusion_matrix.png",
]:
    display(Image(f"{base}/{f}"))
display(pd.read_csv(f"{base}/test/main_metrics.csv"))
display(pd.read_csv(f"{base}/test/defect_size_metrics.csv"))
display(pd.read_csv(f"{base}/test/defect_group_metrics.csv"))


## Optional E1 — full-image resize + pad
Run only as preprocessing ablation; this is not the main E2 configuration.


In [ ]:
!python scripts/training/train_unet.py \
  --epochs 50 --batch-size 2 --grad-accum 2 \
  --augmentation photometric --data-mode resize --run-name e1_resize_seed42
!python scripts/evaluation/evaluate_model.py --model unet --checkpoint results/unet_r18/e1_resize_seed42/checkpoints/best.pt --split val --warmup-batches 5
!python scripts/evaluation/evaluate_model.py --model unet --checkpoint results/unet_r18/e1_resize_seed42/checkpoints/best.pt --split test --warmup-batches 5
